# Getting Started with LLM Hook Analysis Framework

This notebook provides an interactive introduction to the LLM Hook Analysis Framework.

## What You'll Learn
- How to set up hooks on a PyTorch model
- Capturing forward pass activations
- Basic analysis and visualization
- Understanding hook results

## Prerequisites
```bash
pip install -e .
```

In [ ]:
import torch
import torch.nn as nn
from llm_hooks.core import HookManager
from llm_hooks.pytorch import ForwardHook
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
%matplotlib inline

## Step 1: Create a Simple Model

We'll start with a simple feedforward network to understand the basics.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Create model
model = SimpleNet()
print(model)

## Step 2: Set Up Hooks

Now we'll create a `HookManager` and register a `ForwardHook` to capture activations.

In [ ]:
# Create hook manager
manager = HookManager(name="simple_analysis")

# Register a forward hook to capture all layer outputs
forward_hook = ForwardHook(
    compute_stats=True,      # Compute statistics (mean, std, etc.)
    capture_output=True,     # Store actual output tensors
)

manager.register(forward_hook)
print(f"Registered hooks: {[hook.name for hook in manager.hooks]}")

## Step 3: Apply Hooks to Model

The `apply_to_model()` method instruments the model with all registered hooks.

In [ ]:
# Apply hooks to model
manager.apply_to_model(model)
print("✓ Hooks applied successfully!")

## Step 4: Run Inference

Let's run some data through the model. The hooks will automatically capture information.

In [ ]:
# Create sample input
batch_size = 4
input_tensor = torch.randn(batch_size, 128)

# Run inference
with torch.no_grad():
    output = model(input_tensor)

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output.shape}")
print(f"\n✓ Inference complete! Hooks captured data during forward pass.")

## Step 5: Retrieve and Analyze Results

Now let's examine what the hooks captured.

In [ ]:
# Get results
results = manager.get_results()

print(f"Total results captured: {len(results.results)}")
print(f"\nResults by hook type:")
print(results.summary())

## Step 6: Inspect Individual Results

Let's look at the statistics captured for each layer.

In [ ]:
# Examine results for each layer
for result in results.results:
    print(f"\n{'='*60}")
    print(f"Layer: {result.layer_name}")
    print(f"Hook Type: {result.hook_type}")
    
    if 'stats' in result.data:
        stats = result.data['stats']
        print(f"\nActivation Statistics:")
        print(f"  Mean: {stats.get('mean', 'N/A'):.4f}")
        print(f"  Std:  {stats.get('std', 'N/A'):.4f}")
        print(f"  Min:  {stats.get('min', 'N/A'):.4f}")
        print(f"  Max:  {stats.get('max', 'N/A'):.4f}")
        print(f"  Sparsity: {stats.get('sparsity', 'N/A'):.2%}")

## Step 7: Visualize Activation Statistics

Let's create visualizations to better understand the model behavior.

In [ ]:
# Extract statistics for visualization
layer_names = []
means = []
stds = []
sparsities = []

for result in results.results:
    if 'stats' in result.data:
        layer_names.append(result.layer_name)
        stats = result.data['stats']
        means.append(stats.get('mean', 0))
        stds.append(stats.get('std', 0))
        sparsities.append(stats.get('sparsity', 0))

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot means
axes[0].bar(range(len(layer_names)), means, color='steelblue')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Mean Activation')
axes[0].set_title('Activation Means by Layer')
axes[0].set_xticks(range(len(layer_names)))
axes[0].set_xticklabels([name.split('.')[-1] for name in layer_names], rotation=45)

# Plot standard deviations
axes[1].bar(range(len(layer_names)), stds, color='coral')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Std Activation')
axes[1].set_title('Activation Std by Layer')
axes[1].set_xticks(range(len(layer_names)))
axes[1].set_xticklabels([name.split('.')[-1] for name in layer_names], rotation=45)

# Plot sparsity
axes[2].bar(range(len(layer_names)), sparsities, color='mediumseagreen')
axes[2].set_xlabel('Layer')
axes[2].set_ylabel('Sparsity')
axes[2].set_title('Activation Sparsity by Layer')
axes[2].set_xticks(range(len(layer_names)))
axes[2].set_xticklabels([name.split('.')[-1] for name in layer_names], rotation=45)

plt.tight_layout()
plt.show()

## Step 8: Clean Up

Always remove hooks when done to avoid memory leaks.

In [ ]:
# Remove all hooks
manager.remove_all_hooks()
print("✓ All hooks removed successfully!")

## Using Context Manager (Recommended)

The recommended way is to use the context manager, which automatically cleans up:

In [ ]:
# Create fresh model
model = SimpleNet()

# Use context manager
with HookManager() as manager:
    # Register and apply hooks
    manager.register(ForwardHook(compute_stats=True))
    manager.apply_to_model(model)
    
    # Run inference
    with torch.no_grad():
        output = model(torch.randn(4, 128))
    
    # Get results
    results = manager.get_results()
    print(f"Captured {len(results.results)} results")

# Hooks are automatically removed when exiting the context
print("✓ Context manager automatically cleaned up hooks!")

## Next Steps

Now that you understand the basics, try:

1. **02_attention_analysis.ipynb** - Analyze attention patterns in transformers
2. **03_gradient_debugging.ipynb** - Debug training issues with gradient tracking
3. **04_model_pruning.ipynb** - Use Fisher information for model pruning

## Key Takeaways

✅ `HookManager` coordinates all hooks  
✅ `ForwardHook` captures activations and statistics  
✅ Use `apply_to_model()` to instrument the model  
✅ Results are collected automatically during inference  
✅ Always clean up hooks or use context manager  